In [133]:
import numpy as np
import pandas as pd
import os, warnings, datetime, math

from sklearn.preprocessing import LabelEncoder

In [134]:
########################### Helpers
#################################################################################
## -------------------
## Memory Reducer
# :df pandas dataframe to reduce size             # type: pd.DataFrame()
# :verbose                                        # type: bool
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

In [135]:
########################### Vars
#################################################################################
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')

In [136]:
########################### DATA LOAD
#################################################################################
print('Load Data')
train_df = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/data/ieee-fraud-detection/train_transaction.csv')
test_df = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/data/ieee-fraud-detection/test_transaction.csv')
test_df['isFraud'] = 0

train_identity = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/data/ieee-fraud-detection/train_identity.csv')
test_identity = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/data/ieee-fraud-detection/test_identity.csv')

Load Data


/var/folders/qh/10rj2tgd6dld_075lmzprz_h0000gn/T/ipykernel_34104/702616683.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df['isFraud'] = 0


In [137]:
train_df

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [138]:
test_df

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,V331,V332,V333,V334,V335,V336,V337,V338,V339,isFraud
0,3663549,18403224,31.950,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,3663550,18403263,49.000,W,4272,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,3663551,18403310,171.000,W,4476,574.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,3663552,18403310,284.950,W,10989,360.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,3663553,18403317,67.950,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506686,4170235,34214279,94.679,C,13832,375.0,185.0,mastercard,224.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
506687,4170236,34214287,12.173,C,3154,408.0,185.0,mastercard,224.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
506688,4170237,34214326,49.000,W,16661,490.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
506689,4170238,34214337,202.000,W,16621,516.0,150.0,mastercard,224.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [139]:
########################### Base check
#################################################################################
import numpy as np
from pandas.api.types import is_numeric_dtype

dfs = [train_df, test_df, train_identity, test_identity]

for i, df in enumerate(dfs):
    original = df.copy(deep=True)
    reduced = reduce_mem_usage(df)

    for col in reduced.columns:
        if is_numeric_dtype(original[col]) and is_numeric_dtype(reduced[col]):
            a = original[col].to_numpy(dtype="float64", na_value=np.nan)
            b = reduced[col].to_numpy(dtype="float64", na_value=np.nan)

            if not np.allclose(a, b, equal_nan=True):
                reduced[col] = original[col]
                print("Bad transformation:", col)
    dfs[i] = reduced

train_df, test_df, train_identity, test_identity = dfs

Mem. usage decreased to 542.35 Mb (69.4% reduction)
Bad transformation: TransactionAmt
Bad transformation: dist1
Bad transformation: dist2
Bad transformation: C1
Bad transformation: C2
Bad transformation: C4
Bad transformation: C6
Bad transformation: C7
Bad transformation: C8
Bad transformation: C10
Bad transformation: C11
Bad transformation: C12
Bad transformation: C13
Bad transformation: D8
Bad transformation: D9
Bad transformation: V129
Bad transformation: V130
Bad transformation: V131
Bad transformation: V150
Bad transformation: V159
Bad transformation: V205
Bad transformation: V206
Bad transformation: V207
Bad transformation: V208
Bad transformation: V209
Bad transformation: V210
Bad transformation: V266
Bad transformation: V267
Bad transformation: V268
Bad transformation: V269
Bad transformation: V270
Bad transformation: V271
Bad transformation: V272
Bad transformation: V273
Bad transformation: V275
Bad transformation: V309
Bad transformation: V310
Bad transformation: V311
Bad tr

In [140]:
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar as calendar

dates_range = pd.date_range("2017-10-01", "2019-01-01")
us_holidays = calendar().holidays(dates_range.min(), dates_range.max())

TIME_COLS = [
    "DT", "DT_M", "DT_W", "DT_D",
    "DT_hour", "DT_day_week", "DT_day_month", "DT_week_month",
    "is_december", "is_holiday",
    "DT_M_total", "DT_W_total", "DT_D_total"
]

def add_time_features(df):
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop(columns=[c for c in TIME_COLS if c in df.columns], errors="ignore")

    dt = START_DATE + pd.to_timedelta(df["TransactionDT"], unit="s")
    iso_week = dt.dt.isocalendar().week.astype("int16")

    feats = pd.DataFrame({
        "DT": dt,
        "DT_M": (((dt.dt.year - 2017) * 12 + dt.dt.month)).astype("int8"),
        "DT_W": (((dt.dt.year - 2017) * 52 + iso_week)).astype("int16"),
        "DT_D": (((dt.dt.year - 2017) * 365 + dt.dt.dayofyear)).astype("int16"),
        "DT_hour": dt.dt.hour.astype("int8"),
        "DT_day_week": dt.dt.dayofweek.astype("int8"),
        "DT_day_month": dt.dt.day.astype("int8"),
        "DT_week_month": (((dt.dt.day - 1) // 7) + 1).astype("int8"),
        "is_december": (dt.dt.month == 12).astype("int8"),
        "is_holiday": dt.dt.normalize().isin(us_holidays).astype("int8"),
    }, index=df.index)

    return pd.concat([df, feats], axis=1)

train_df = add_time_features(train_df)
test_df = add_time_features(test_df)

for col in ["DT_M", "DT_W", "DT_D"]:
    vals = np.concatenate([train_df[col].to_numpy(), test_df[col].to_numpy()])
    fq = pd.Series(vals).value_counts().to_dict()
    train_df[f"{col}_total"] = train_df[col].map(fq)
    test_df[f"{col}_total"] = test_df[col].map(fq)

In [141]:
train_df

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,DT_D,DT_hour,DT_day_week,DT_day_month,DT_week_month,is_december,is_holiday,DT_M_total,DT_W_total,DT_D_total
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,335,0,4,1,1,1,0,137321,12093,5122
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,335,0,4,1,1,1,0,137321,12093,5122
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,335,0,4,1,1,1,0,137321,12093,5122
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,335,0,4,1,1,1,0,137321,12093,5122
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,335,0,4,1,1,1,0,137321,12093,5122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,516,23,3,31,5,0,0,89326,10288,2754
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,516,23,3,31,5,0,0,89326,10288,2754
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,516,23,3,31,5,0,0,89326,10288,2754
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,516,23,3,31,5,0,0,89326,10288,2754


In [142]:
test_df

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,DT_D,DT_hour,DT_day_week,DT_day_month,DT_week_month,is_december,is_holiday,DT_M_total,DT_W_total,DT_D_total
0,3663549,18403224,31.950,W,10409,111.0,150.0,visa,226.0,debit,...,547,0,6,1,1,0,0,78430,2243,2243
1,3663550,18403263,49.000,W,4272,111.0,150.0,visa,226.0,debit,...,547,0,6,1,1,0,0,78430,2243,2243
2,3663551,18403310,171.000,W,4476,574.0,150.0,visa,226.0,debit,...,547,0,6,1,1,0,0,78430,2243,2243
3,3663552,18403310,284.950,W,10989,360.0,150.0,visa,166.0,debit,...,547,0,6,1,1,0,0,78430,2243,2243
4,3663553,18403317,67.950,W,18018,452.0,150.0,mastercard,117.0,debit,...,547,0,6,1,1,0,0,78430,2243,2243
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506686,4170235,34214279,94.679,C,13832,375.0,185.0,mastercard,224.0,debit,...,729,23,6,30,5,1,0,116398,28119,2514
506687,4170236,34214287,12.173,C,3154,408.0,185.0,mastercard,224.0,debit,...,729,23,6,30,5,1,0,116398,28119,2514
506688,4170237,34214326,49.000,W,16661,490.0,150.0,visa,226.0,debit,...,729,23,6,30,5,1,0,116398,28119,2514
506689,4170238,34214337,202.000,W,16621,516.0,150.0,mastercard,224.0,debit,...,729,23,6,30,5,1,0,116398,28119,2514


In [143]:
########################### card4, card6, ProductCD
#################################################################################
# Converting Strings to ints(or floats if nan in column) using frequency encoding
# We will be able to use these columns as category or as numerical feature

for col in ['card4', 'card6', 'ProductCD']:
    print('Encoding', col)
    temp_df = pd.concat([train_df[[col]], test_df[[col]]])
    col_encoded = temp_df[col].value_counts().to_dict()
    train_df[col] = train_df[col].map(col_encoded)
    test_df[col]  = test_df[col].map(col_encoded)
    print(col_encoded)

Encoding card4
{'visa': 719649, 'mastercard': 347386, 'american express': 16009, 'discover': 9524}
Encoding card6
{'debit': 824959, 'credit': 267648, 'debit or credit': 30, 'charge card': 16}
Encoding ProductCD
{'W': 800657, 'C': 137785, 'R': 73346, 'H': 62397, 'S': 23046}


In [144]:
m_cols = [c for c in train_df.columns if c[0] == "M" and c[1:].isdigit()]
print(m_cols)
for c in m_cols:
    print(f"\n=== {c} ===")
    print(train_df[c].value_counts(dropna=False))

['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']

=== M1 ===
M1
T      319415
NaN    271100
F          25
Name: count, dtype: int64

=== M2 ===
M2
T      285468
NaN    271100
F       33972
Name: count, dtype: int64

=== M3 ===
M3
NaN    271100
T      251731
F       67709
Name: count, dtype: int64

=== M4 ===
M4
NaN    281444
M0     196405
M2      59865
M1      52826
Name: count, dtype: int64

=== M5 ===
M5
NaN    350482
F      132491
T      107567
Name: count, dtype: int64

=== M6 ===
M6
F      227856
T      193324
NaN    169360
Name: count, dtype: int64

=== M7 ===
M7
NaN    346265
F      211374
T       32901
Name: count, dtype: int64

=== M8 ===
M8
NaN    346252
F      155251
T       89037
Name: count, dtype: int64

=== M9 ===
M9
NaN    346252
T      205656
F       38632
Name: count, dtype: int64


In [145]:
########################### M columns
#################################################################################
# Converting Strings to ints(or floats if nan in column)

for col in ['M1','M2','M3','M5','M6','M7','M8','M9']:
    train_df[col] = train_df[col].map({'T':1, 'F':0})
    test_df[col]  = test_df[col].map({'T':1, 'F':0})

for col in ['M4']:
    print('Encoding', col)
    temp_df = pd.concat([train_df[[col]], test_df[[col]]])
    col_encoded = temp_df[col].value_counts().to_dict()
    train_df[col] = train_df[col].map(col_encoded)
    test_df[col]  = test_df[col].map(col_encoded)
    print(col_encoded)

Encoding M4
{'M0': 357789, 'M2': 122947, 'M1': 97306}


In [146]:
train_df

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,DT_D,DT_hour,DT_day_week,DT_day_month,DT_week_month,is_december,is_holiday,DT_M_total,DT_W_total,DT_D_total
0,2987000,0,86400,68.50,800657,13926,NaN,150.0,9524.0,142.0,...,335,0,4,1,1,1,0,137321,12093,5122
1,2987001,0,86401,29.00,800657,2755,404.0,150.0,347386.0,102.0,...,335,0,4,1,1,1,0,137321,12093,5122
2,2987002,0,86469,59.00,800657,4663,490.0,150.0,719649.0,166.0,...,335,0,4,1,1,1,0,137321,12093,5122
3,2987003,0,86499,50.00,800657,18132,567.0,150.0,347386.0,117.0,...,335,0,4,1,1,1,0,137321,12093,5122
4,2987004,0,86506,50.00,62397,4497,514.0,150.0,347386.0,102.0,...,335,0,4,1,1,1,0,137321,12093,5122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,800657,6550,NaN,150.0,719649.0,226.0,...,516,23,3,31,5,0,0,89326,10288,2754
590536,3577536,0,15811049,39.50,800657,10444,225.0,150.0,347386.0,224.0,...,516,23,3,31,5,0,0,89326,10288,2754
590537,3577537,0,15811079,30.95,800657,12037,595.0,150.0,347386.0,224.0,...,516,23,3,31,5,0,0,89326,10288,2754
590538,3577538,0,15811088,117.00,800657,7826,481.0,150.0,347386.0,224.0,...,516,23,3,31,5,0,0,89326,10288,2754


In [147]:
test_df

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,DT_D,DT_hour,DT_day_week,DT_day_month,DT_week_month,is_december,is_holiday,DT_M_total,DT_W_total,DT_D_total
0,3663549,18403224,31.950,800657,10409,111.0,150.0,719649.0,226.0,824959.0,...,547,0,6,1,1,0,0,78430,2243,2243
1,3663550,18403263,49.000,800657,4272,111.0,150.0,719649.0,226.0,824959.0,...,547,0,6,1,1,0,0,78430,2243,2243
2,3663551,18403310,171.000,800657,4476,574.0,150.0,719649.0,226.0,824959.0,...,547,0,6,1,1,0,0,78430,2243,2243
3,3663552,18403310,284.950,800657,10989,360.0,150.0,719649.0,166.0,824959.0,...,547,0,6,1,1,0,0,78430,2243,2243
4,3663553,18403317,67.950,800657,18018,452.0,150.0,347386.0,117.0,824959.0,...,547,0,6,1,1,0,0,78430,2243,2243
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506686,4170235,34214279,94.679,137785,13832,375.0,185.0,347386.0,224.0,824959.0,...,729,23,6,30,5,1,0,116398,28119,2514
506687,4170236,34214287,12.173,137785,3154,408.0,185.0,347386.0,224.0,824959.0,...,729,23,6,30,5,1,0,116398,28119,2514
506688,4170237,34214326,49.000,800657,16661,490.0,150.0,719649.0,226.0,824959.0,...,729,23,6,30,5,1,0,116398,28119,2514
506689,4170238,34214337,202.000,800657,16621,516.0,150.0,347386.0,224.0,824959.0,...,729,23,6,30,5,1,0,116398,28119,2514


In [148]:
id_cols = [idc for idc in train_identity.columns if idc.startswith("id")]

for idc in id_cols:
    print(f"\n=== {idc} ===")
    print(train_identity[idc].value_counts(dropna=False))


=== id_01 ===
id_01
-5.0     82170
 0.0     19555
-10.0    11257
-20.0    11211
-15.0     5674
         ...  
-47.0        1
-82.0        1
-48.0        1
-63.0        1
-51.0        1
Name: count, Length: 77, dtype: int64

=== id_02 ===
id_02
NaN         3361
1102.0        11
696.0         10
1120.0         9
1083.0         9
            ... 
253305.0       1
145955.0       1
172059.0       1
632381.0       1
55528.0        1
Name: count, Length: 115656, dtype: int64

=== id_03 ===
id_03
 NaN     77909
 0.0     63903
 1.0       863
 3.0       668
 2.0       421
 5.0       109
 4.0       100
 6.0        64
-5.0        33
-6.0        31
-4.0        21
-7.0        21
-10.0       17
-8.0        14
-2.0        12
-1.0        12
-3.0         8
-9.0         6
-11.0        6
 7.0         4
 9.0         3
-12.0        3
-13.0        3
 10.0        1
 8.0         1
Name: count, dtype: int64

=== id_04 ===
id_04
 NaN     77909
 0.0     65739
-5.0       132
-6.0        98
-8.0        64
-4.0    

In [149]:
train_identity

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144228,3577521,-15.0,145955.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 66.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,F3111 Build/33.3.A.1.97
144229,3577526,-5.0,172059.0,NaN,NaN,1.0,-5.0,NaN,NaN,NaN,...,chrome 55.0 for android,32.0,855x480,match_status:2,T,F,T,F,mobile,A574BL Build/NMF26F
144230,3577529,-20.0,632381.0,NaN,NaN,-1.0,-36.0,NaN,NaN,NaN,...,chrome 65.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,Moto E (4) Plus Build/NMA26.42-152
144231,3577531,-5.0,55528.0,0.0,0.0,0.0,-7.0,NaN,NaN,0.0,...,chrome 66.0,24.0,2560x1600,match_status:2,T,F,T,F,desktop,MacOS


In [150]:
test_identity

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,id-01,id-02,id-03,id-04,id-05,id-06,id-07,id-08,id-09,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663586,-45.0,280290.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,MYA-L13 Build/HUAWEIMYA-L13
1,3663588,0.0,3579.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 67.0 for android,24.0,1280x720,match_status:2,T,F,T,T,mobile,LGLS676 Build/MXB48T
2,3663597,-5.0,185210.0,NaN,NaN,1.0,0.0,NaN,NaN,NaN,...,ie 11.0 for tablet,NaN,NaN,NaN,F,T,T,F,desktop,Trident/7.0
3,3663601,-45.0,252944.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,MYA-L13 Build/HUAWEIMYA-L13
4,3663602,-95.0,328680.0,NaN,NaN,7.0,-33.0,NaN,NaN,NaN,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,SM-G9650 Build/R16NW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141902,4170230,-20.0,473365.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,chrome 71.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,SM-J700M
141903,4170233,-5.0,489917.0,0.0,0.0,-4.0,-32.0,NaN,NaN,0.0,...,chrome 71.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,SM-J320M
141904,4170234,-5.0,110081.0,NaN,NaN,22.0,-31.0,NaN,NaN,NaN,...,mobile safari 10.0,32.0,1334x750,match_status:2,T,F,F,T,mobile,iOS Device
141905,4170236,-45.0,266704.0,NaN,NaN,-3.0,-10.0,NaN,NaN,NaN,...,chrome 43.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,ALE-L23 Build/HuaweiALE-L23


In [151]:
# rename only columns that start with "id-"
test_identity = test_identity.rename(
    columns=lambda c: c.replace("id-", "id_", 1) if c.startswith("id-") else c
)

In [152]:
test_identity = test_identity.reindex(columns=train_identity.columns)

In [153]:
print(set(train_identity.columns) - set(test_identity.columns)) # should be empty

set()


In [154]:
########################### Identity columns
#################################################################################

def minify_identity_df(df):

    df['id_12'] = df['id_12'].map({'Found':1, 'NotFound':0})
    df['id_15'] = df['id_15'].map({'New':2, 'Found':1, 'Unknown':0})
    df['id_16'] = df['id_16'].map({'Found':1, 'NotFound':0})

    df['id_23'] = df['id_23'].map({'TRANSPARENT':4, 'IP_PROXY':3, 'IP_PROXY:ANONYMOUS':2, 'IP_PROXY:HIDDEN':1})

    df['id_27'] = df['id_27'].map({'Found':1, 'NotFound':0})
    df['id_28'] = df['id_28'].map({'New':2, 'Found':1})

    df['id_29'] = df['id_29'].map({'Found':1, 'NotFound':0})

    df['id_35'] = df['id_35'].map({'T':1, 'F':0})
    df['id_36'] = df['id_36'].map({'T':1, 'F':0})
    df['id_37'] = df['id_37'].map({'T':1, 'F':0})
    df['id_38'] = df['id_38'].map({'T':1, 'F':0})

    df['id_34'] = df['id_34'].fillna(':0')
    df['id_34'] = df['id_34'].apply(lambda x: x.split(':')[1]).astype(np.int8)
    df['id_34'] = np.where(df['id_34']==0, np.nan, df['id_34'])

    df['id_33'] = df['id_33'].fillna('0x0')
    df['id_33_0'] = df['id_33'].apply(lambda x: x.split('x')[0]).astype(int)
    df['id_33_1'] = df['id_33'].apply(lambda x: x.split('x')[1]).astype(int)
    df['id_33'] = np.where(df['id_33']=='0x0', np.nan, df['id_33'])

    df['DeviceType'].map({'desktop':1, 'mobile':0})
    return df

train_identity = minify_identity_df(train_identity)
test_identity = minify_identity_df(test_identity)

for col in ['id_33']:
    train_identity[col] = train_identity[col].fillna('unseen_before_label')
    test_identity[col]  = test_identity[col].fillna('unseen_before_label')

    le = LabelEncoder()
    le.fit(list(train_identity[col])+list(test_identity[col]))
    train_identity[col] = le.transform(train_identity[col])
    test_identity[col]  = le.transform(test_identity[col])

In [155]:
id_cols = [idc for idc in train_identity.columns if idc.startswith("id")]

for idc in id_cols:
    print(f"\n=== {idc} ===")
    print(train_identity[idc].value_counts(dropna=False))


=== id_01 ===
id_01
-5.0     82170
 0.0     19555
-10.0    11257
-20.0    11211
-15.0     5674
         ...  
-47.0        1
-82.0        1
-48.0        1
-63.0        1
-51.0        1
Name: count, Length: 77, dtype: int64

=== id_02 ===
id_02
NaN         3361
1102.0        11
696.0         10
1120.0         9
1083.0         9
            ... 
253305.0       1
145955.0       1
172059.0       1
632381.0       1
55528.0        1
Name: count, Length: 115656, dtype: int64

=== id_03 ===
id_03
 NaN     77909
 0.0     63903
 1.0       863
 3.0       668
 2.0       421
 5.0       109
 4.0       100
 6.0        64
-5.0        33
-6.0        31
-4.0        21
-7.0        21
-10.0       17
-8.0        14
-2.0        12
-1.0        12
-3.0         8
-9.0         6
-11.0        6
 7.0         4
 9.0         3
-12.0        3
-13.0        3
 10.0        1
 8.0         1
Name: count, dtype: int64

=== id_04 ===
id_04
 NaN     77909
 0.0     65739
-5.0       132
-6.0        98
-8.0        64
-4.0    

In [156]:
train_identity

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,id_33_0,id_33_1
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,267,2.0,1.0,0.0,1.0,1.0,mobile,SAMSUNG SM-G892A Build/NRD90M,2220,1080
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,79,1.0,1.0,0.0,0.0,1.0,mobile,iOS Device,1334,750
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,460,NaN,0.0,0.0,1.0,1.0,desktop,Windows,0,0
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,460,NaN,0.0,0.0,1.0,1.0,desktop,NaN,0,0
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,67,2.0,1.0,0.0,1.0,1.0,desktop,MacOS,1280,800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144228,3577521,-15.0,145955.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,460,NaN,0.0,0.0,1.0,0.0,mobile,F3111 Build/33.3.A.1.97,0,0
144229,3577526,-5.0,172059.0,NaN,NaN,1.0,-5.0,NaN,NaN,NaN,...,448,2.0,1.0,0.0,1.0,0.0,mobile,A574BL Build/NMF26F,855,480
144230,3577529,-20.0,632381.0,NaN,NaN,-1.0,-36.0,NaN,NaN,NaN,...,460,NaN,0.0,0.0,1.0,0.0,mobile,Moto E (4) Plus Build/NMA26.42-152,0,0
144231,3577531,-5.0,55528.0,0.0,0.0,0.0,-7.0,NaN,NaN,0.0,...,315,2.0,1.0,0.0,1.0,0.0,desktop,MacOS,2560,1600


In [157]:
test_identity

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,id_33_0,id_33_1
0,3663586,-45.0,280290.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,460,NaN,0.0,0.0,1.0,0.0,mobile,MYA-L13 Build/HUAWEIMYA-L13,0,0
1,3663588,0.0,3579.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,61,2.0,1.0,0.0,1.0,1.0,mobile,LGLS676 Build/MXB48T,1280,720
2,3663597,-5.0,185210.0,NaN,NaN,1.0,0.0,NaN,NaN,NaN,...,460,NaN,0.0,1.0,1.0,0.0,desktop,Trident/7.0,0,0
3,3663601,-45.0,252944.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,460,NaN,0.0,0.0,1.0,0.0,mobile,MYA-L13 Build/HUAWEIMYA-L13,0,0
4,3663602,-95.0,328680.0,NaN,NaN,7.0,-33.0,NaN,NaN,NaN,...,460,NaN,0.0,0.0,1.0,0.0,mobile,SM-G9650 Build/R16NW,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141902,4170230,-20.0,473365.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,460,NaN,0.0,0.0,1.0,0.0,mobile,SM-J700M,0,0
141903,4170233,-5.0,489917.0,0.0,0.0,-4.0,-32.0,NaN,NaN,0.0,...,460,NaN,0.0,0.0,1.0,0.0,mobile,SM-J320M,0,0
141904,4170234,-5.0,110081.0,NaN,NaN,22.0,-31.0,NaN,NaN,NaN,...,79,2.0,1.0,0.0,0.0,1.0,mobile,iOS Device,1334,750
141905,4170236,-45.0,266704.0,NaN,NaN,-3.0,-10.0,NaN,NaN,NaN,...,460,NaN,0.0,0.0,1.0,0.0,mobile,ALE-L23 Build/HuaweiALE-L23,0,0


In [158]:
########################### Deltas

for df in [train_df, test_df]:
    for col in ['D'+str(i) for i in range(1,16) if i!=9]:
        new_col = 'uid_td_'+str(col)
        df[new_col] = df[col].fillna(0).astype(int)
        df[new_col] = df[new_col].apply(lambda x: pd.Timedelta(x, unit='D'))
        df[new_col] = (df['DT'] - df[new_col]).dt.date
        df[new_col] = df[new_col].astype(str)
        df[new_col] = np.where(df[col].isna(), np.nan, df[new_col])

In [159]:
train_df

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,uid_td_D5,uid_td_D6,uid_td_D7,uid_td_D8,uid_td_D10,uid_td_D11,uid_td_D12,uid_td_D13,uid_td_D14,uid_td_D15
0,2987000,0,86400,68.50,800657,13926,NaN,150.0,9524.0,142.0,...,NaN,NaN,NaN,NaN,2017-11-18,2017-11-18,NaN,NaN,NaN,2017-12-01
1,2987001,0,86401,29.00,800657,2755,404.0,150.0,347386.0,102.0,...,NaN,NaN,NaN,NaN,2017-12-01,NaN,NaN,NaN,NaN,2017-12-01
2,2987002,0,86469,59.00,800657,4663,490.0,150.0,719649.0,166.0,...,NaN,NaN,NaN,NaN,2017-12-01,2017-01-20,NaN,NaN,NaN,2017-01-20
3,2987003,0,86499,50.00,800657,18132,567.0,150.0,347386.0,117.0,...,2017-12-01,NaN,NaN,NaN,2017-09-08,NaN,NaN,NaN,NaN,2017-08-12
4,2987004,0,86506,50.00,62397,4497,514.0,150.0,347386.0,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,800657,6550,NaN,150.0,719649.0,226.0,...,NaN,NaN,NaN,NaN,2018-04-05,2018-04-05,NaN,NaN,NaN,2018-04-05
590536,3577536,0,15811049,39.50,800657,10444,225.0,150.0,347386.0,224.0,...,NaN,NaN,NaN,NaN,2018-05-31,2018-05-31,NaN,NaN,NaN,2018-05-31
590537,3577537,0,15811079,30.95,800657,12037,595.0,150.0,347386.0,224.0,...,NaN,NaN,NaN,NaN,2018-05-31,2018-05-31,NaN,NaN,NaN,2018-05-31
590538,3577538,0,15811088,117.00,800657,7826,481.0,150.0,347386.0,224.0,...,2018-05-31,NaN,NaN,NaN,2018-05-09,2018-05-09,NaN,NaN,NaN,2018-05-09


In [160]:
test_df

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,uid_td_D5,uid_td_D6,uid_td_D7,uid_td_D8,uid_td_D10,uid_td_D11,uid_td_D12,uid_td_D13,uid_td_D14,uid_td_D15
0,3663549,18403224,31.950,800657,10409,111.0,150.0,719649.0,226.0,824959.0,...,2018-06-04,NaN,NaN,NaN,2017-05-09,2017-12-10,NaN,NaN,NaN,2017-05-18
1,3663550,18403263,49.000,800657,4272,111.0,150.0,719649.0,226.0,824959.0,...,2018-06-24,NaN,NaN,NaN,2017-11-12,2016-10-05,NaN,NaN,NaN,2016-10-05
2,3663551,18403310,171.000,800657,4476,574.0,150.0,719649.0,226.0,824959.0,...,2018-06-21,NaN,NaN,NaN,2018-02-15,2018-02-15,NaN,NaN,NaN,2018-03-26
3,3663552,18403310,284.950,800657,10989,360.0,150.0,719649.0,166.0,824959.0,...,2018-05-21,NaN,NaN,NaN,2017-11-01,2017-11-01,NaN,NaN,NaN,2017-11-01
4,3663553,18403317,67.950,800657,18018,452.0,150.0,347386.0,117.0,824959.0,...,2018-07-01,NaN,NaN,NaN,2018-06-09,2018-06-09,NaN,NaN,NaN,2018-06-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506686,4170235,34214279,94.679,137785,13832,375.0,185.0,347386.0,224.0,824959.0,...,NaN,2018-12-30,NaN,NaN,2018-12-30,NaN,2018-12-30,NaN,NaN,2018-12-30
506687,4170236,34214287,12.173,137785,3154,408.0,185.0,347386.0,224.0,824959.0,...,2018-12-14,2018-12-14,2018-12-14,NaN,2018-12-30,NaN,2018-12-14,2018-12-30,2018-12-30,2018-12-14
506688,4170237,34214326,49.000,800657,16661,490.0,150.0,719649.0,226.0,824959.0,...,NaN,NaN,NaN,NaN,2018-12-30,2018-12-30,NaN,NaN,NaN,2018-12-30
506689,4170238,34214337,202.000,800657,16621,516.0,150.0,347386.0,224.0,824959.0,...,NaN,NaN,NaN,NaN,2018-12-30,2018-12-30,NaN,NaN,NaN,2018-12-30


In [161]:
########################### Final check
#################################################################################

frames = {
    "train_df": train_df,
    "test_df": test_df,
    "train_identity": train_identity,
    "test_identity": test_identity,
}

for name, df in frames.items():
    original = df.copy(deep=True)
    reduced = reduce_mem_usage(df.copy())

    for col in reduced.columns:
        a = original[col]
        b = reduced[col]

        # Compare only numeric columns safely
        if is_numeric_dtype(a) and is_numeric_dtype(b):
            a_np = a.to_numpy(dtype="float64", na_value=np.nan)
            b_np = b.to_numpy(dtype="float64", na_value=np.nan)

            if not np.allclose(a_np, b_np, equal_nan=True):
                reduced[col] = a
                print(f"Bad transformation: {name}.{col}")

    frames[name] = reduced

train_df = frames["train_df"]
test_df = frames["test_df"]
train_identity = frames["train_identity"]
test_identity = frames["test_identity"]

Mem. usage decreased to 585.15 Mb (25.1% reduction)
Bad transformation: train_df.TransactionAmt
Bad transformation: train_df.dist1
Bad transformation: train_df.dist2
Bad transformation: train_df.C1
Bad transformation: train_df.C2
Bad transformation: train_df.C4
Bad transformation: train_df.C6
Bad transformation: train_df.C7
Bad transformation: train_df.C8
Bad transformation: train_df.C10
Bad transformation: train_df.C11
Bad transformation: train_df.C12
Bad transformation: train_df.C13
Bad transformation: train_df.D8
Bad transformation: train_df.D9
Bad transformation: train_df.V129
Bad transformation: train_df.V130
Bad transformation: train_df.V131
Bad transformation: train_df.V150
Bad transformation: train_df.V159
Bad transformation: train_df.V205
Bad transformation: train_df.V206
Bad transformation: train_df.V207
Bad transformation: train_df.V208
Bad transformation: train_df.V209
Bad transformation: train_df.V210
Bad transformation: train_df.V266
Bad transformation: train_df.V267
Bad 

In [162]:
########################### Export
#################################################################################

train_df.to_pickle('pkl_exported_files/train_transaction.pkl')
test_df.to_pickle('pkl_exported_files/test_transaction.pkl')

train_identity.to_pickle('pkl_exported_files/train_identity.pkl')
test_identity.to_pickle('pkl_exported_files/test_identity.pkl')

In [163]:
########################### Full minification for fast tests
#################################################################################
for df in [train_df, test_df, train_identity, test_identity]:
    df = reduce_mem_usage(df)

Mem. usage decreased to 585.15 Mb (20.6% reduction)
Mem. usage decreased to 509.80 Mb (16.2% reduction)
Mem. usage decreased to 15.54 Mb (5.0% reduction)
Mem. usage decreased to 15.29 Mb (5.0% reduction)


In [164]:
########################### Export
#################################################################################

train_df.to_pickle('pkl_exported_files/train_transaction_mini.pkl')
test_df.to_pickle('pkl_exported_files/test_transaction_mini.pkl')

train_identity.to_pickle('pkl_exported_files/train_identity_mini.pkl')
test_identity.to_pickle('pkl_exported_files/test_identity_mini.pkl')

In [165]:
########################### Export
#################################################################################
import pickle
possible_goups = [['V1', 'V2', 'V6', 'V7', 'V8', 'V9'],
 ['V1', 'V2', 'V3', 'V6', 'V7', 'V8', 'V9'],
 ['V2', 'V3', 'V6', 'V7', 'V8', 'V9'],
 ['V4', 'V5'],
 ['V10', 'V11'],
 ['V12', 'V13'],
 ['V14', 'V65'],
 ['V15', 'V16', 'V33', 'V34', 'V57', 'V58', 'V79', 'V94'],
 ['V15', 'V16', 'V33', 'V34', 'V57'],
 ['V17', 'V18', 'V21', 'V22'],
 ['V19', 'V20'],
 ['V17', 'V18', 'V21', 'V22', 'V63', 'V84'],
 ['V23', 'V24'],
 ['V25', 'V26'],
 ['V27', 'V28', 'V68', 'V89'],
 ['V29', 'V30', 'V69', 'V90', 'V91'],
 ['V29', 'V30', 'V70', 'V90', 'V91'],
 ['V31', 'V32', 'V50', 'V71', 'V92', 'V93'],
 ['V31', 'V32', 'V92'],
 ['V15', 'V16', 'V33', 'V34', 'V51', 'V94'],
 ['V15', 'V16', 'V33', 'V34'],
 ['V35', 'V36'],
 ['V37', 'V38'],['V39', 'V40'],
 ['V41', 'V46', 'V47'],
 ['V42', 'V43', 'V84'],
 ['V42', 'V43'],
 ['V44', 'V45'],
 ['V48', 'V49'],
 ['V31', 'V50', 'V71', 'V92'],
 ['V33', 'V51', 'V52', 'V73', 'V94'],
 ['V51', 'V52'],
 ['V53', 'V54'],
 ['V15', 'V16', 'V57', 'V58', 'V73', 'V79', 'V94'],
 ['V15', 'V57', 'V58', 'V79'],
 ['V59', 'V60', 'V63'],
 ['V59', 'V60'],
 ['V61', 'V62'],
 ['V21', 'V59', 'V63', 'V64', 'V84'],
 ['V63', 'V64'],
 ['V66', 'V67'],
 ['V29', 'V69', 'V70', 'V90'],
 ['V30', 'V69', 'V70', 'V90', 'V91'],
 ['V31', 'V50', 'V71', 'V72', 'V92', 'V93'],
 ['V71', 'V72', 'V92', 'V93'],
 ['V51', 'V57', 'V73', 'V74', 'V94'],
 ['V73', 'V74'],
 ['V75', 'V76'],
 ['V80', 'V81', 'V84'],
 ['V80', 'V81'],
 ['V82', 'V83'],
 ['V21', 'V42', 'V63', 'V80', 'V84', 'V85'],
 ['V84', 'V85'],['V86', 'V87'],
 ['V29', 'V30', 'V69', 'V70', 'V90', 'V91'],
 ['V31', 'V32', 'V50', 'V71', 'V72', 'V92', 'V93'],
 ['V31', 'V71', 'V72', 'V92', 'V93'],
 ['V15', 'V33', 'V51', 'V57', 'V73', 'V94'],
 ['V101',
  'V102',
  'V103',
  'V143',
  'V167',
  'V168',
  'V177',
  'V178',
  'V179',
  'V279',
  'V280',
  'V293',
  'V295',
  'V322',
  'V323',
  'V324',
  'V95',
  'V96',
  'V97'],['V101',
  'V102',
  'V103',
  'V143',
  'V167',
  'V168',
  'V177',
  'V178',
  'V179',
  'V279',
  'V280',
  'V293',
  'V294',
  'V295',
  'V322',
  'V323',
  'V324',
  'V95',
  'V96',
  'V97'],
 ['V105', 'V106', 'V296', 'V298', 'V299', 'V329', 'V330'],
 ['V105', 'V106', 'V298', 'V299', 'V329', 'V330'],
 ['V111', 'V113'],
 ['V126', 'V128', 'V132', 'V134'],
 ['V127', 'V128', 'V133', 'V134'],
 ['V126', 'V127', 'V128', 'V132', 'V133', 'V134', 'V332'],
 ['V129', 'V266', 'V269', 'V309', 'V334'],
 ['V130', 'V310'],
 ['V131', 'V312'],['V126', 'V128', 'V132', 'V133', 'V134'],
 ['V127', 'V128', 'V132', 'V133', 'V134'],
 ['V126', 'V127', 'V128', 'V132', 'V133', 'V134', 'V318', 'V332'],
 ['V136', 'V137'],
 ['V101',
  'V102',
  'V103',
  'V143',
  'V167',
  'V177',
  'V178',
  'V179',
  'V279',
  'V280',
  'V293',
  'V295',
  'V322',
  'V323',
  'V324',
  'V95',
  'V96',
  'V97'],
 ['V144', 'V145', 'V150', 'V151'],
 ['V148', 'V149', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158'],
 ['V144', 'V145', 'V150', 'V151', 'V152'],
 ['V151', 'V152'],
 ['V161', 'V163'],
 ['V162', 'V163'],
 ['V161', 'V162', 'V163'],['V101',
  'V102',
  'V103',
  'V167',
  'V168',
  'V177',
  'V178',
  'V179',
  'V279',
  'V280',
  'V293',
  'V295',
  'V322',
  'V323',
  'V324',
  'V95',
  'V96',
  'V97'],
 ['V176', 'V190', 'V199', 'V228', 'V246', 'V257'],
 ['V180', 'V182', 'V183'],
 ['V181', 'V328'],
 ['V180', 'V182', 'V183', 'V330'],
 ['V186', 'V191', 'V196'],
 ['V187', 'V192'],
 ['V187', 'V192', 'V193'],
 ['V192', 'V193', 'V196'],
 ['V194', 'V197'],
 ['V195', 'V198'],
 ['V186', 'V191', 'V193', 'V196'],
 ['V202', 'V204', 'V211', 'V213'],['V203', 'V204', 'V212'],
 ['V202', 'V203', 'V204', 'V213'],
 ['V202', 'V211', 'V213'],
 ['V203', 'V212', 'V213'],
 ['V202', 'V204', 'V211', 'V212', 'V213'],
 ['V214', 'V276', 'V337'],
 ['V215', 'V216', 'V277', 'V278', 'V338', 'V339'],
 ['V217', 'V219', 'V231', 'V233'],
 ['V218', 'V219', 'V232', 'V233'],
 ['V217', 'V218', 'V219', 'V231', 'V232', 'V233'],
 ['V222', 'V230'],
 ['V224', 'V225'],
 ['V229', 'V230', 'V258'],
 ['V222', 'V229', 'V230', 'V258'],
 ['V236', 'V237'],
 ['V238', 'V239'],
 ['V240', 'V241', 'V247', 'V252', 'V260'],
 ['V242', 'V244'],
 ['V245', 'V259'],
 ['V240', 'V241', 'V247', 'V249', 'V252'],
 ['V248', 'V249', 'V254'],
 ['V247', 'V248', 'V249', 'V252'],
 ['V250', 'V251'],
 ['V248', 'V254'],
 ['V255', 'V256'],
 ['V240', 'V241', 'V260'],
 ['V263', 'V265', 'V273', 'V274', 'V275'],
 ['V264', 'V265'],
 ['V263', 'V264', 'V265'],
 ['V129', 'V266', 'V269', 'V309', 'V334', 'V336'],['V268', 'V336'],
 ['V270', 'V272'],
 ['V263', 'V273', 'V274', 'V275'],
 ['V291', 'V292'],
 ['V102', 'V280', 'V294', 'V295', 'V323', 'V96'],
 ['V105', 'V296', 'V298', 'V299', 'V329'],
 ['V105', 'V106', 'V296', 'V298', 'V299', 'V329'],
 ['V105', 'V106', 'V296', 'V298', 'V299', 'V330'],
 ['V300', 'V301'],
 ['V302', 'V304'],
 ['V303', 'V304'],
 ['V302', 'V303', 'V304'],
 ['V306', 'V308', 'V316', 'V318'],
 ['V307', 'V308', 'V317'],
 ['V306', 'V307', 'V308', 'V318'],
 ['V313', 'V315'],
 ['V306', 'V316', 'V318'],
 ['V307', 'V317', 'V318'],
 ['V134', 'V306', 'V308', 'V316', 'V317', 'V318'],
 ['V320', 'V321'],
 ['V326', 'V327'],
 ['V105', 'V106', 'V296', 'V298', 'V329', 'V330'],
 ['V105', 'V106', 'V183', 'V299', 'V329', 'V330'],
 ['V331', 'V332', 'V333'],
 ['V128', 'V134', 'V331', 'V332', 'V333'],
 ['V335', 'V336'],
 ['V266', 'V268', 'V269', 'V334', 'V335', 'V336']]

# save
with open('pkl_exported_files/possible_groups.pickle', 'wb') as f:
    pickle.dump(possible_goups, f, pickle.HIGHEST_PROTOCOL)

In [166]:
# load
with open("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/data/ieee-fraud-detection/pkl_exported_files/possible_groups.pickle", "rb") as f:
    possible_groups_loaded = pickle.load(f)

In [167]:
print(type(possible_groups_loaded), len(possible_groups_loaded))
print(possible_groups_loaded[:2])

<class 'list'> 151
[['V1', 'V2', 'V6', 'V7', 'V8', 'V9'], ['V1', 'V2', 'V3', 'V6', 'V7', 'V8', 'V9']]
